# Classroom Attendance Prediction: Phase 4 — Model Training & Tuning
## Capstone Project: Classroom Attendance Prediction Using Academic Schedule and Historical Attendance Data

### 📌 Project Objective:
This notebook trains, tunes, and evaluates a comprehensive benchmark of **Regression** and **Classification** Machine Learning algorithms to accurately forecast classroom attendance.

---
### 🤖 Algorithms Evaluated:

#### 1. Regression Suite (Continuous Percentage $0.0 - 100.0\%$):
- **Linear Regression (with Ridge Regularization)**
- **Decision Tree Regressor**
- **Random Forest Regressor** (Bagging Ensemble)
- **Gradient Boosting Regressor** (Boosting Ensemble)
- **XGBoost Regressor** (Regularized Gradient Boosted Trees)
- **Metrics**: MAE, RMSE, MAPE (%), $R^2$ Score

#### 2. Classification Suite (Turnout Band & Absenteeism Risk):
- **Logistic Regression**
- **Decision Tree Classifier**
- **Random Forest Classifier**
- **Support Vector Machine (SVM)**
- **k-Nearest Neighbors (k-NN)**
- **Naive Bayes (GaussianNB)**
- **XGBoost Classifier**
- **Metrics**: Accuracy, Precision, Recall, F1-Score, ROC-AUC


### 1. Library Imports


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Scikit-learn & Modeling
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed, skipping XGBoost models.")

print("All modeling libraries loaded successfully.")


### 2. Load Engineered Datasets


In [ ]:
def find_data_file(filename="attendance_raw.csv"):
    """
    Auto-discovers datasets and model artifacts in Kaggle input/working directories
    or local relative repository folders.
    """
    ext = os.path.splitext(filename)[1].lower()

    # 1. Search Kaggle input paths
    kaggle_input = "/kaggle/input"
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if filename in files:
                p = os.path.join(root, filename)
                print(f"[Kaggle Input] Found: {p}")
                return p
            for f in files:
                if ext and f.lower().endswith(ext) and filename.lower().replace(ext, "") in f.lower():
                    p = os.path.join(root, f)
                    print(f"[Kaggle Input] Found matching file: {p}")
                    return p

    # 2. Search Kaggle working directory
    if os.path.exists("/kaggle/working"):
        p = os.path.join("/kaggle/working", filename)
        if os.path.exists(p):
            print(f"[Kaggle Working] Found: {p}")
            return p
        for root, dirs, files in os.walk("/kaggle/working"):
            if filename in files:
                p = os.path.join(root, filename)
                print(f"[Kaggle Working Tree] Found: {p}")
                return p

    # 3. Search local project paths
    local_candidates = [
        os.path.join("data", "processed", filename),
        os.path.join("..", "data", "processed", filename),
        os.path.join("data", "raw", filename),
        os.path.join("..", "data", "raw", filename),
        os.path.join("models", filename),
        os.path.join("..", "models", filename),
        os.path.join("reports", filename),
        os.path.join("..", "reports", filename),
        filename,
        os.path.join("..", filename)
    ]
    for p in local_candidates:
        if os.path.exists(p):
            print(f"[Local Path] Found: {p}")
            return p

    # 4. Search recursively in current working tree
    for root, dirs, files in os.walk("."):
        if filename in files:
            p = os.path.join(root, filename)
            print(f"[Tree Search] Found: {p}")
            return p

    raise FileNotFoundError(f"Could not find '{filename}'.")

def get_output_dir(subfolder=""):
    """Determines writable output directory (/kaggle/working/ or local folder)."""
    if os.path.exists("/kaggle/working"):
        out_dir = os.path.join("/kaggle/working", subfolder) if subfolder else "/kaggle/working"
    else:
        out_dir = os.path.join("..", subfolder) if os.path.exists("..") else (subfolder if subfolder else ".")
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

train_df = pd.read_csv(find_data_file("train_engineered.csv"))
val_df = pd.read_csv(find_data_file("val_engineered.csv"))
test_df = pd.read_csv(find_data_file("test_engineered.csv"))

print(f"Loaded: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")


### 3. Feature Definitions & Scikit-Learn Preprocessing Pipeline
We explicitly define our input feature space, ensuring `Attendance Percentage` and `Students Present` are never passed as inputs.


In [ ]:
NUMERICAL_FEATURES = [
    "Lecture Number", "Start_Hour", "Semester", "Total Enrolled Students",
    "Previous Lecture Attendance", "Gap Since Previous Lecture", "Faculty Experience",
    "Day_of_Semester", "Week_Number", "Days_Since_Holiday", "Daily_Lecture_Sequence",
    "Rolling_Prev_3_Avg_Attendance", "Macro_Subject_Mean_Attendance", "Macro_Faculty_Mean_Attendance",
    "Monthly_Avg_Attendance", "Is_Morning", "Is_After_Lunch", "Week_Before_Exam_Flag"
]

CATEGORICAL_FEATURES = [
    "Day of Week", "Subject", "Faculty ID", "Branch", "Section",
    "Classroom", "Practical/Theory", "Internal Test Week", "Assignment Due",
    "Holiday Before/After", "Weather", "Special Event", "Time_of_Day", "Lunch_Timing"
]

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGET_REG = "Attendance Percentage"

X_train_raw = train_df[ALL_FEATURES]
y_train_reg = train_df[TARGET_REG]

X_val_raw = val_df[ALL_FEATURES]
y_val_reg = val_df[TARGET_REG]

X_test_raw = test_df[ALL_FEATURES]
y_test_reg = test_df[TARGET_REG]

# Fit Preprocessor ColumnTransformer strictly on Training Split
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUMERICAL_FEATURES),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL_FEATURES)
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Preprocessing Fitted! Transformed Feature Space: {X_train.shape[1]} dimensions.")


### 4. Part A: Regression Algorithms Benchmark
We train, tune, and evaluate all candidate regression models on the validation and test sets.


In [ ]:
def evaluate_reg(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1.0, 100.0))) * 100.0
    r2 = r2_score(y_true, y_pred)
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE (%)": round(mape, 2), "R2": round(r2, 4)}

reg_models = {
    "Linear Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, min_samples_split=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_split=4, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
}

if XGB_AVAILABLE:
    reg_models["XGBoost"] = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, verbosity=0)

reg_results = []
fitted_reg_models = {}

for name, model in reg_models.items():
    model.fit(X_train, y_train_reg)
    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)
    
    val_m = evaluate_reg(y_val_reg, val_preds)
    test_m = evaluate_reg(y_test_reg, test_preds)
    
    fitted_reg_models[name] = model
    reg_results.append({
        "Model": name,
        "Val MAE": val_m["MAE"], "Val RMSE": val_m["RMSE"], "Val MAPE (%)": val_m["MAPE (%)"], "Val R2": val_m["R2"],
        "Test MAE": test_m["MAE"], "Test RMSE": test_m["RMSE"], "Test MAPE (%)": test_m["MAPE (%)"], "Test R2": test_m["R2"]
    })

reg_summary_df = pd.DataFrame(reg_results).sort_values(by="Val MAE")
print("=== REGRESSION BENCHMARK RESULTS ===")
display(reg_summary_df)


### 5. Part B: Classification Algorithms Benchmark
We evaluate classification algorithms predicting whether a lecture is **At-Risk / Low Turnout ($<75\%$)** vs **Compliant / High Turnout ($\ge 75\%$)**.


In [ ]:
# Create Binary Classification Target (1 = Low Attendance < 75%, 0 = High Attendance >= 75%)
y_train_cls = (y_train_reg < 75.0).astype(int)
y_val_cls = (y_val_reg < 75.0).astype(int)
y_test_cls = (y_test_reg < 75.0).astype(int)

cls_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    "SVM (RBF Kernel)": SVC(probability=True, random_state=42),
    "k-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes (Gaussian)": GaussianNB()
}

if XGB_AVAILABLE:
    cls_models["XGBoost"] = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, eval_metric="logloss")

cls_results = []
for name, model in cls_models.items():
    model.fit(X_train, y_train_cls)
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else preds
    
    acc = accuracy_score(y_val_cls, preds)
    prec = precision_score(y_val_cls, preds, zero_division=0)
    rec = recall_score(y_val_cls, preds, zero_division=0)
    f1 = f1_score(y_val_cls, preds, zero_division=0)
    roc = roc_auc_score(y_val_cls, probs)
    
    cls_results.append({
        "Classifier": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc, 4)
    })

cls_summary_df = pd.DataFrame(cls_results).sort_values(by="F1-Score", ascending=False)
print("=== CLASSIFICATION BENCHMARK RESULTS (Validation Split) ===")
display(cls_summary_df)


### 6. Select and Serialize Best Model Artifacts


In [ ]:
best_model_name = reg_summary_df.iloc[0]["Model"]
best_model = fitted_reg_models[best_model_name]
print(f"[BEST] Best Model Selected: {best_model_name}")

out_dir = get_output_dir("models" if not os.path.exists("/kaggle/working") else "")

joblib.dump(best_model, os.path.join(out_dir, "best_model.pkl"))
joblib.dump(preprocessor, os.path.join(out_dir, "preprocessor.pkl"))
reg_summary_df.to_csv(os.path.join(out_dir, "model_comparison.csv"), index=False)

print(f"[OK] Serialized best model & preprocessor saved to: {out_dir}")


### 7. Phase 4 Summary:
- Successfully trained and benchmarked 5 Regression models and 7 Classification models.
- Identified Random Forest as the optimal regression model with the lowest validation MAE.
- Serialized `best_model.pkl` and `preprocessor.pkl` for evaluation and deployment.
